# Step 6 — Person Vault Construction
**Tough Talks · Phase 3**

Goal: prove Gemma 4 E2B (text-only) can read a conversation transcript
and build / update a PersonVault profile for a *recurring counterparty*
matching `data/schemas/person_vault.schema.json`.

PersonVault is the **mirror of TalkDNA** — TalkDNA captures how the
**user** communicates, PersonVault captures how a specific other person
(`other` speaker in the transcript) communicates. Later steps (persona
simulation, premortem, pulse) consume the PersonVault profile so
practice and prediction are calibrated to who the user is actually
talking to.

**Hybrid design** — same hybrid split as Step 05:
- Code computes deterministic numerics (`other_turn_count`,
  `avg_other_turn_words`, deflection candidate phrases,
  `interruption_of_user_rate` when timing is available) so the model
  never has to count.
- A single prompt-based call to Gemma 4 produces the qualitative
  fields (`communication_style` enum, `emotional_triggers`,
  `de_escalation_keys`, curated `common_deflections`,
  `responds_best_to`, optional `cultural_context`).

Same two-shot retry pattern as Steps 04 / 05 — greedy first, light
sampling on JSON / validation failure.

Notebook is a thin driver; all logic lives in
`backend/core/_runtime/person_vault.py`. Step 11 fills in
`relationship_pulse` later; Step 06 leaves that field off and the
schema treats it as optional so the v1 / v2 profiles still validate.

**What "done" looks like for this step**
1. Text-only Gemma 4 loads (`AutoModelForCausalLM` via `LoadConfig(multimodal=False)`).
2. `compute_person_metrics()` returns sensible numerics on both sample conversations (counterparty turn counts, deflection candidates).
3. `analyze_person_vault()` returns a PersonVault dict for conversation A (`version=1`, generated `person_id`).
4. `analyze_person_vault()` with `prior_profile=profile_v1` returns an updated dict for conversation B (`version=2`, `conversation_count=2`, same `person_id`, accumulated `emotional_triggers` / `de_escalation_keys` / `common_deflections` across both conversations).
5. Every result validates against `data/schemas/person_vault.schema.json` (required fields, communication_style enum, relationship_type enum, accumulating list shape).

**Why accumulation (not replacement) for the qualitative lists**
For TalkDNA, the model dropping prior identifiers across conversations
was tracked as [[hypothesis-talkdna-identifier-accumulation]] — open
because the right behaviour depends on whether downstream consumers
benefit from continuity. For PersonVault the answer is clearly **yes**:
losing an `emotional_trigger` observed in conversation 1 just because
conversation 2 didn't re-demonstrate it would defeat the point of a
per-person rolling profile. So the runtime accumulates here by default
(deduped, capped at 8 items). This is the same hypothesis tested in
the opposite direction.

In [2]:
# ── 0. Install / upgrade dependencies ─────────────────────────────────
# Text-only path — no audio libs required. Same rule as the audio steps:
# bump only transformers + accelerate on Colab / Kaggle (bumping torch
# breaks the pre-installed torchvision / CUDA pairing). After this first
# run, RESTART THE KERNEL before continuing if you actually upgraded
# transformers — the already-imported version won't pick up the change.

!pip install -q -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 110.6 MB/s eta 0:00:0000:010:01


In [12]:
# ── 1. Locate (or fetch) the repo, put it on sys.path ─────────────────────
# Same shim as Steps 01–05 — auto-clones / refreshes on Colab / Kaggle.
# After the refresh we drop any cached `backend.*` modules so subsequent
# imports pick up the freshly-pulled code instead of whatever this kernel
# imported earlier in the session.

import os, pathlib, subprocess, sys

REPO_URL  = "https://github.com/EhsanFarazmand/tough_talks.git"
REPO_NAME = "tough_talks"

def _looks_like_repo(p: pathlib.Path) -> bool:
    return (p / "backend" / "core" / "_runtime").is_dir()

def _scan_for_repo() -> pathlib.Path | None:
    cwd = pathlib.Path.cwd()
    for parent in [cwd, *cwd.parents]:
        if _looks_like_repo(parent):
            return parent
    for base in (pathlib.Path("/content"), pathlib.Path("/kaggle/working")):
        candidate = base / REPO_NAME
        if _looks_like_repo(candidate):
            return candidate
    return None

def _refresh(target: pathlib.Path) -> None:
    if not (target / ".git").is_dir():
        return
    print(f"Refreshing {target} from origin")
    subprocess.run(["git", "-C", str(target), "fetch", "--depth", "1", "origin"],
                   capture_output=True, check=False)
    subprocess.run(["git", "-C", str(target), "reset", "--hard", "FETCH_HEAD"],
                   capture_output=True, check=False)

REPO_ROOT = _scan_for_repo()
if REPO_ROOT is None:
    base = next((b for b in (pathlib.Path("/content"), pathlib.Path("/kaggle/working")) if b.is_dir()),
                pathlib.Path.cwd())
    target = base / REPO_NAME
    print(f"Cloning {REPO_URL} -> {target}")
    result = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(target)],
                            capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError("git clone failed:\n" + result.stderr)
    REPO_ROOT = target
else:
    _refresh(REPO_ROOT)

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

_stale = [m for m in list(sys.modules) if m == "backend" or m.startswith("backend.")]
for _m in _stale:
    del sys.modules[_m]
if _stale:
    print(f"Cleared {len(_stale)} cached backend.* module(s) from sys.modules")

print(f"Repo root: {REPO_ROOT}")

Refreshing /content/tough_talks from origin
Cleared 13 cached backend.* module(s) from sys.modules
Repo root: /content/tough_talks


In [13]:
# ── 2. Imports ──────────────────────────────────────────────────
import json
from pathlib import Path

import torch

from backend.core._runtime import (
    ALLOWED_COMMUNICATION_STYLES,
    ALLOWED_RELATIONSHIP_TYPES,
    DEFAULT_MODEL_ID,
    JsonParseError,
    LoadConfig,
    PersonVaultAnalysisError,
    PersonVaultConfig,
    analyze_person_vault,
    compute_person_metrics,
    load_model,
)

In [5]:
# ── 3. Configuration ─────────────────────────────────────────────────

MODEL_ID    = DEFAULT_MODEL_ID                # google/gemma-4-E2B-it
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
SCHEMA_PATH = Path(REPO_ROOT) / "data" / "schemas" / "person_vault.schema.json"

# The counterparty we are profiling — same transcripts as Step 05, but
# now we are watching `other` instead of `user`. Pretend the user has
# tagged this person as a colleague called "Jamie" in the UI.
PERSON_NAME       = "Jamie"
RELATIONSHIP_TYPE = "colleague"

# Conversation A — same nine-turn argument transcript as Steps 03 / 04 / 05.
# `other` (Jamie) deflects with "I told you", "Don't blame me",
# "Don't put this on me", "You weren't there", and eventually concedes
# with a constructive offer ("staging numbers, flag the gaps"). This is
# the source of the v1 PersonVault profile.
CONVERSATION_A = [
    {"speaker": "user",  "text": "I just wanted to check on the report — it was due Monday and it's already Thursday.",
     "emotion": {"primary": "frustration", "intensity": 0.7}},
    {"speaker": "other", "text": "I told you on Tuesday the data team hadn't delivered. I can't just make numbers up.",
     "emotion": {"primary": "defensiveness", "intensity": 0.8}},
    {"speaker": "user",  "text": "Sorry, I kind of forgot you mentioned that. But you still should escalate — you don't just sit on it.",
     "emotion": {"primary": "frustration", "intensity": 0.75}},
    {"speaker": "other", "text": "I did escalate. You weren't in the meeting. Don't blame me for your missed message.",
     "emotion": {"primary": "defensiveness", "intensity": 0.75}},
    {"speaker": "user",  "text": "I guess I just can't be in every meeting. That's why we have email — sorry, I should've checked it.",
     "emotion": {"primary": "frustration", "intensity": 0.6}},
    {"speaker": "other", "text": "I sent two emails. You replied to neither. Don't put this on me.",
     "emotion": {"primary": "frustration", "intensity": 0.7}},
    {"speaker": "user",  "text": "Okay, fair. My bad, I missed them. But we still have a problem to solve tonight.",
     "emotion": {"primary": "frustration", "intensity": 0.55, "concession_made": True}},
    {"speaker": "other", "text": "I have partial numbers from the staging tables. We can present those and flag the gaps.",
     "emotion": {"primary": "openness", "intensity": 0.5}},
    {"speaker": "user",  "text": "Good. Let's regroup at six. And next time, just call me directly — I'm sorry I missed the emails.",
     "emotion": {"primary": "neutral", "intensity": 0.3}},
]

# Conversation B — a shorter follow-up the next morning. Jamie is more
# constructive (proposes a clear escalation trigger) but the underlying
# pattern still shows ("That's aggressive given..." is a softened
# deflection). This drives the v2 profile via the incremental path and
# tests that the v1 triggers / de-escalation keys are accumulated, not
# discarded.
CONVERSATION_B = [
    {"speaker": "user",  "text": "I want to set a hard deadline for the data team — Wednesday EOD, no exceptions."},
    {"speaker": "other", "text": "That's aggressive given the staging tables aren't done."},
    {"speaker": "user",  "text": "Sorry, I know it's tight. But I just need a number we can defend to finance."},
    {"speaker": "other", "text": "Then commit to escalating Tuesday morning if blockers are still open."},
    {"speaker": "user",  "text": "Fair. I'll send the calendar block today."},
]

print(f"Model              : {MODEL_ID}")
print(f"Device             : {DEVICE}")
print(f"Person             : {PERSON_NAME} ({RELATIONSHIP_TYPE})")
print(f"Conv A             : {len(CONVERSATION_A)} turns")
print(f"Conv B             : {len(CONVERSATION_B)} turns")
print(f"Comm styles        : {ALLOWED_COMMUNICATION_STYLES}")
print(f"Relationship types : {ALLOWED_RELATIONSHIP_TYPES}")

Model              : google/gemma-4-E2B-it
Device             : cuda
Person             : Jamie (colleague)
Conv A             : 9 turns
Conv B             : 5 turns
Comm styles        : ('direct', 'indirect', 'passive_aggressive', 'avoidant', 'collaborative', 'dominant', 'assertive', 'empathetic', 'defensive')
Relationship types : ('partner', 'parent', 'sibling', 'friend', 'manager', 'report', 'colleague', 'other')


In [6]:
# ── 4. Deterministic metrics sanity-check (no model needed) ────────────────
# `compute_person_metrics` is pure Python — we can inspect what the
# code-side accounting looks like for the counterparty before paying
# the model-loading cost. This block is also useful for debugging the
# deflection regex on any new transcript.

for label, turns in (("A", CONVERSATION_A), ("B", CONVERSATION_B)):
    m = compute_person_metrics(turns)
    print(f"Conversation {label}:")
    print(f"  other_turn_count         : {m.other_turn_count}")
    print(f"  total_turn_count         : {m.total_turn_count}")
    print(f"  avg_other_turn_words     : {m.avg_other_turn_words:.1f}")
    print(f"  deflection_candidates    : {m.deflection_candidates}")
    print(f"  interruption_of_user_rate: {m.interruption_of_user_rate}")

Conversation A:
  other_turn_count         : 4
  total_turn_count         : 9
  avg_other_turn_words     : 15.0
  deflection_candidates    : ['i told you', "don't blame me", "you weren't there", "don't put this on me"]
  interruption_of_user_rate: None
Conversation B:
  other_turn_count         : 2
  total_turn_count         : 5
  avg_other_turn_words     : 9.5
  deflection_candidates    : []
  interruption_of_user_rate: None


In [7]:
# ── 5. Load the text-only processor + model ────────────────────────────
# PersonVault reasons over WORDS — same rationale as TalkDNA. The
# `multimodal=False` (default) path loads `AutoModelForCausalLM`, which
# is lighter on VRAM and slightly faster than the multimodal class.

processor, model = load_model(LoadConfig(model_id=MODEL_ID))
n_params = sum(p.numel() for p in model.parameters()) / 1e9
print(f"Model loaded ({n_params:.1f}B parameters, on {model.device})")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

Model loaded (5.1B parameters, on cuda:0)


In [14]:
# ── 6. Conversation A → first PersonVault profile (version 1) ─────────────
# The runtime renders the prompt from (a) the counterparty's display
# name + relationship type, (b) the transcript with emotion tags,
# (c) the deterministic metrics block, and (d) a "no prior" sentinel
# for the prior-profile block. Greedy first, light sampling on parse
# failure (same retry policy as Steps 04 / 05).

profile_v1 = analyze_person_vault(
    processor,
    model,
    CONVERSATION_A,
    cfg=PersonVaultConfig(
        name=PERSON_NAME,
        relationship_type=RELATIONSHIP_TYPE,
    ),
)
print(json.dumps(profile_v1, indent=2))

{
  "person_id": "person_cf528d57b789",
  "name": "Jamie",
  "relationship_type": "colleague",
  "version": 1,
  "conversation_count": 1,
  "profile": {
    "communication_style": "defensive",
    "emotional_triggers": [
      "asking for report status",
      "suggesting escalation",
      "mentioning missed emails"
    ],
    "de_escalation_keys": [
      "acknowledging missed communication",
      "agreeing on next steps"
    ],
    "common_deflections": [
      "I told you",
      "Don't blame me",
      "Don't put this on me"
    ],
    "responds_best_to": "Direct accountability for past actions and clear, actionable next steps. Responds well when the user accepts responsibility for communication gaps."
  },
  "updated_at": "2026-05-14T09:10:40.044429+00:00"
}


In [15]:
# ── 7. Conversation B with prior_profile=v1 → updated profile (version 2) ───
# This is the incremental path. The runtime bumps `version` to 2,
# increments `conversation_count`, keeps the same `person_id`, and
# **accumulates** the qualitative lists across conversations (deduped,
# capped at 8 items). The qualitative fields evolve via the prompt,
# which now sees the v1 profile as prior context.

profile_v2 = analyze_person_vault(
    processor,
    model,
    CONVERSATION_B,
    cfg=PersonVaultConfig(
        name=PERSON_NAME,
        relationship_type=RELATIONSHIP_TYPE,
        prior_profile=profile_v1,
    ),
)
print(json.dumps(profile_v2, indent=2))

{
  "person_id": "person_cf528d57b789",
  "name": "Jamie",
  "relationship_type": "colleague",
  "version": 2,
  "conversation_count": 2,
  "profile": {
    "communication_style": "assertive",
    "emotional_triggers": [
      "asking for report status",
      "suggesting escalation",
      "mentioning missed emails",
      "setting a hard deadline",
      "stating a need for a number"
    ],
    "de_escalation_keys": [
      "acknowledging missed communication",
      "agreeing on next steps",
      "acknowledging the tightness",
      "proposing a clear escalation path"
    ],
    "common_deflections": [
      "I told you",
      "Don't blame me",
      "Don't put this on me",
      "staging tables aren't done"
    ],
    "responds_best_to": "Clear, actionable next steps tied to specific milestones. Responds well when the user accepts responsibility for communication gaps."
  },
  "updated_at": "2026-05-14T09:12:17.614889+00:00"
}


In [16]:
# ── 8. Schema validation + results table ───────────────────────────────
# Hand-rolled validator (no jsonschema dep — matches Steps 04 / 05).
# Per profile:
#   - required top-level fields present
#   - profile.communication_style in the schema enum (when present)
#   - relationship_type in the schema enum (when present)
#   - emotional_triggers / de_escalation_keys / common_deflections are
#     lists of plain strings
# Cross-profile (v1 → v2):
#   - person_id stable
#   - version bumped, conversation_count incremented
#   - qualitative lists accumulated (every v1 item still in v2)

schema = json.loads(SCHEMA_PATH.read_text(encoding="utf-8"))
SCHEMA_REQUIRED    = schema["required"]
COMM_STYLE_ENUM    = schema["properties"]["profile"]["properties"]["communication_style"]["enum"]
RELATIONSHIP_ENUM  = schema["properties"]["relationship_type"]["enum"]


def _validate_person_vault(profile: dict) -> list[str]:
    errs: list[str] = []
    for k in SCHEMA_REQUIRED:
        if k not in profile:
            errs.append(f"missing top-level key {k!r}")
    pf = profile.get("profile")
    if not isinstance(pf, dict):
        errs.append("profile is not an object")
        return errs
    style = pf.get("communication_style")
    if style is not None and style not in COMM_STYLE_ENUM:
        errs.append(f"communication_style {style!r} not in {COMM_STYLE_ENUM}")
    rt = profile.get("relationship_type")
    if rt is not None and rt not in RELATIONSHIP_ENUM:
        errs.append(f"relationship_type {rt!r} not in {RELATIONSHIP_ENUM}")
    for list_key in ("emotional_triggers", "de_escalation_keys", "common_deflections"):
        items = pf.get(list_key, [])
        if not isinstance(items, list):
            errs.append(f"profile.{list_key} is not a list")
            continue
        for item in items:
            if not isinstance(item, str) or not item.strip():
                errs.append(f"profile.{list_key} contains non-string / empty: {item!r}")
    for str_key in ("responds_best_to", "cultural_context"):
        v = pf.get(str_key)
        if v is not None and not isinstance(v, str):
            errs.append(f"profile.{str_key} not a string: {v!r}")
    return errs


profiles = [("v1 (conversation A)", profile_v1), ("v2 (after conversation B)", profile_v2)]
per_profile_errors = [(label, _validate_person_vault(p)) for label, p in profiles]
n_clean = sum(1 for _, errs in per_profile_errors if not errs)

# Accumulation check — every v1 qualitative-list entry should survive
# into v2 (deduped, casing-insensitive).
def _set_lower(items):
    return {x.strip().lower() for x in (items or []) if isinstance(x, str) and x.strip()}

v1_triggers = _set_lower(profile_v1.get("profile", {}).get("emotional_triggers"))
v2_triggers = _set_lower(profile_v2.get("profile", {}).get("emotional_triggers"))
v1_de_esc   = _set_lower(profile_v1.get("profile", {}).get("de_escalation_keys"))
v2_de_esc   = _set_lower(profile_v2.get("profile", {}).get("de_escalation_keys"))
v1_defl     = _set_lower(profile_v1.get("profile", {}).get("common_deflections"))
v2_defl     = _set_lower(profile_v2.get("profile", {}).get("common_deflections"))

triggers_accumulated = v1_triggers.issubset(v2_triggers) or not v1_triggers
de_esc_accumulated   = v1_de_esc.issubset(v2_de_esc) or not v1_de_esc
defl_accumulated     = v1_defl.issubset(v2_defl) or not v1_defl

checks: list[tuple[str, bool, str]] = [
    ("text_model_loaded",        True,                                              f"{n_params:.1f}B params on {model.device}"),
    ("profile_v1_built",         isinstance(profile_v1, dict),                       f"version={profile_v1.get('version')}"),
    ("profile_v2_built",         isinstance(profile_v2, dict),                       f"version={profile_v2.get('version')}"),
    ("version_incremented",      profile_v2.get("version") == 2,                     f"v1={profile_v1.get('version')} -> v2={profile_v2.get('version')}"),
    ("conversation_count_2",     profile_v2.get("conversation_count") == 2,          f"{profile_v2.get('conversation_count')}"),
    ("person_id_stable",         profile_v1.get("person_id") == profile_v2.get("person_id"),
                                                                                     f"{profile_v1.get('person_id')}"),
    ("schema_valid_per_profile", n_clean == len(profiles),                           f"{n_clean}/{len(profiles)} clean"),
    ("triggers_accumulated",     triggers_accumulated,                               f"v1={sorted(v1_triggers)} ⊆ v2={sorted(v2_triggers)}"),
    ("de_escalation_accumulated", de_esc_accumulated,                                f"v1={sorted(v1_de_esc)} ⊆ v2={sorted(v2_de_esc)}"),
    ("deflections_accumulated",  defl_accumulated,                                   f"v1={sorted(v1_defl)} ⊆ v2={sorted(v2_defl)}"),
]

print("=" * 72)
print("STEP 6 RESULTS — Gemma 4 Person Vault construction")
print("=" * 72)
all_ok = True
for name, ok, note in checks:
    icon = "PASS" if ok else "FAIL"
    print(f"[{icon}]  {name:28s}  {note}")
    if not ok:
        all_ok = False

print("\nProfile evolution:")
for label, p in profiles:
    pf = p.get("profile", {})
    print(f"  {label}")
    print(f"    person_id              : {p.get('person_id')}")
    print(f"    name / relationship    : {p.get('name')} ({p.get('relationship_type')})")
    print(f"    communication_style    : {pf.get('communication_style')}")
    print(f"    emotional_triggers     : {pf.get('emotional_triggers')}")
    print(f"    de_escalation_keys     : {pf.get('de_escalation_keys')}")
    print(f"    common_deflections     : {pf.get('common_deflections')}")
    print(f"    responds_best_to       : {pf.get('responds_best_to')}")
    if pf.get("cultural_context"):
        print(f"    cultural_context       : {pf.get('cultural_context')}")

if any(errs for _, errs in per_profile_errors):
    print("\nSchema errors:")
    for label, errs in per_profile_errors:
        if not errs:
            continue
        print(f"  {label}:")
        for line in errs:
            print(f"    - {line}")

print()
print("OVERALL:", "READY FOR STEP 7" if all_ok else "FIX FAILURES ABOVE")

STEP 6 RESULTS — Gemma 4 Person Vault construction
[PASS]  text_model_loaded             5.1B params on cuda:0
[PASS]  profile_v1_built              version=1
[PASS]  profile_v2_built              version=2
[PASS]  version_incremented           v1=1 -> v2=2
[PASS]  conversation_count_2          2
[PASS]  person_id_stable              person_cf528d57b789
[PASS]  schema_valid_per_profile      2/2 clean
[PASS]  triggers_accumulated          v1=['asking for report status', 'mentioning missed emails', 'suggesting escalation'] ⊆ v2=['asking for report status', 'mentioning missed emails', 'setting a hard deadline', 'stating a need for a number', 'suggesting escalation']
[PASS]  de_escalation_accumulated     v1=['acknowledging missed communication', 'agreeing on next steps'] ⊆ v2=['acknowledging missed communication', 'acknowledging the tightness', 'agreeing on next steps', 'proposing a clear escalation path']
[PASS]  deflections_accumulated       v1=["don't blame me", "don't put this on me", 